# Context-Faithfulness Check

**Is the AI answer faithful to your source?** Paste a source, a policy, a regulation clause, a contract section, any ground-truth, and ask a question. The tool has the model answer using that source, then runs one lightweight pass that reads the answer back against the source and tells you: does every claim trace to the source, did it add anything, does it contradict the source. A faithfulness read to help your review. About a minute, free, no signup.

This notebook is **pre-filled with a demonstration default** so your first run shows the read *running* on a real example. The point is to watch the mechanism work, not to watch it pass.

A free, own-key tool by [pysolvr / LLM Prover](https://llmprover.pysolvr.com).

---

> **This is a read, NOT a certification.** It comments on faithfulness; it will never tell you an answer is "compliant" or "verified". It is a single lightweight pass that can be inconsistent and can miss things, a first look to guide *your* review. If your profession requires you to verify AI output, this does not discharge that duty. You own your review; this is an instrument to assist it.

> **Free tool, provided as-is, no warranty.** It uses *your own* API key and calls the provider directly. **Your key never leaves your machine / your Colab account and is never sent to pysolvr.** You are responsible for your own usage and costs. Outputs are informational, not advice. pysolvr is not liable for decisions made from these results.

**Key safety:** this notebook never asks you to type a key into a cell (see step 2). Keys live in Colab Secrets or your shell environment, so a saved or shared copy contains no secrets.

## 1. Setup

Pulls the tool from the repo and installs the provider SDKs. `openai` also covers Gemini, Grok, Together, and Groq via their OpenAI-compatible endpoints.

In [ ]:
!pip -q install openai>=1.0 anthropic>=0.30
!wget -q -O faithfulness_core.py https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/faithfulness/faithfulness_core.py
!wget -q -O prices.json          https://raw.githubusercontent.com/dAIvdMercer/llmprover-tools/main/tools/faithfulness/prices.json
print('ready')

## 2. Add your API key (do NOT type it into a cell)

**Never paste an API key directly into a notebook cell.** A key typed into a cell gets saved into the notebook file, your Colab history, and any copy you share. Store it *outside* the notebook instead. The cell below only *reads* it.

### On Colab (recommended), use the Secrets panel
1. Click the **key icon** (Secrets) in the left sidebar.
2. **Add a secret** for your model's provider, using the exact name:

   | Secret name | Models it unlocks |
   |-------------|-------------------|
   | `OPENAI_API_KEY` | `gpt-*`, `o3`/`o4-*` |
   | `ANTHROPIC_API_KEY` | `claude*` |
   | `GEMINI_API_KEY` | `gemini*` |
   | `XAI_API_KEY` | `grok*` |
   | `TOGETHER_API_KEY` | `<org>/<model>` |
   | `GROQ_API_KEY` | `qwen/*` |

3. Toggle **Notebook access** on for the secret. One provider's key is enough if the answering and reading models are from the same provider.

### On local Jupyter, environment variable
Set it in your shell *before* launching Jupyter: `export OPENAI_API_KEY=sk-...` then `jupyter lab`.

Then run the cell below. **Nothing is sent to pysolvr; your key stays on your machine / in your Colab account.**

In [ ]:
# Loads keys from Colab Secrets (preferred) or the environment. Contains no key itself.
import os

_KEY_NAMES = ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'XAI_API_KEY', 'TOGETHER_API_KEY', 'GROQ_API_KEY']
try:
    from google.colab import userdata  # Colab only
    for _name in _KEY_NAMES:
        try:
            _val = userdata.get(_name)
            if _val:
                os.environ[_name] = _val
        except Exception:
            pass  # secret not set / notebook access off -- skip
except ImportError:
    pass  # not on Colab -- keys come from the shell environment

_loaded = [n for n in _KEY_NAMES if os.environ.get(n)]
print('keys loaded for:', _loaded if _loaded else 'NONE -- add one via the Colab Secrets panel (key icon, left) with Notebook access ON, or set an env var. See cell above.')

## 3. Your source and your question

The cell below is pre-filled with a demonstration: a short leave policy as the `SOURCE` and a question that invites the model to add detail the policy does not state. Run it as-is to watch the read flag anything added or contradicted.

- `SOURCE` -- the ground-truth text you are checking against (paste a policy, clause, guideline).
- `QUESTION` -- what you ask, answered using the source.
- `MODEL` answers; `JUDGE_MODEL` does the faithfulness read (can be the same).

Use the provider's own model-ID strings (the tool is model-agnostic). To find what exists and what it costs: **[llmprover.pysolvr.com/models](https://llmprover.pysolvr.com/models/)** and **[llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/)**.

To check **your own**: paste your `SOURCE` between the triple quotes and set `QUESTION` to yours.

In [ ]:
MODEL       = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o-mini'   # the model that does the faithfulness read (can be the same)

QUESTION = 'Can an employee carry unused annual leave into the next year, and how is leave paid out on exit?'

SOURCE = """
Annual Leave Policy. Full-time employees receive 20 days of paid annual leave per calendar year.
Leave is booked at least two weeks in advance and approved by the employee's manager.
"""

## 4. Run the faithfulness read

The model answers your question using the source, then one reading pass reports a faithfulness read: appears faithful / possible additions / possible contradiction / not in source, with the specific claims it could not find in your source.

**Read the reminder in the output.** This is a single-pass read to help your review, not a compliance certification. It can be inconsistent and can miss things, and it does not discharge any professional duty to verify AI output.

In [ ]:
from faithfulness_core import faithfulness_core, format_result

result = faithfulness_core(source=SOURCE, question=QUESTION, model=MODEL, judge_model=JUDGE_MODEL)
print(format_result(result))

---

**Prices are a dated snapshot, not a live feed.** For the live, authoritative table see [llmprover.pysolvr.com/api-pricing](https://llmprover.pysolvr.com/api-pricing/).

If you're interested in more free tools to test and measure different aspects of an AI pipeline, see our [free AI tools](https://llmprover.pysolvr.com/tools/). This is a single-pass read. For faithfulness you can rely on, rubric-scored, consistency-checked, with a diagnostic and an audit trail, across your whole regulation or policy corpus, see [Features](https://llmprover.pysolvr.com/features/?tool=faithfulness).